# Vector stores and semantic search

1. librerias necesarias para la ejecucion de la actividad

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from IPython.display import display

## Part I: Basic vector store implementation


2. tomando como referencia el código del github

In [12]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def _normalize(self, vectors: np.ndarray) -> np.ndarray:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return vectors / norms

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [document.text for document in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True
        )
        new_embeddings = self._normalize(new_embeddings.astype(np.float32))

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            show_progress_bar=False
        )
        query_embedding = self._normalize(query_embedding.astype(np.float32))[0]

        scores = self.embeddings @ query_embedding
        top_k = min(top_k, len(self.documents))
        top_indices = np.argsort(scores)[::-1][:top_k]

        return [
            SearchResult(score=float(scores[index]), document=self.documents[index])
            for index in top_indices
        ]

### Animal Fun Facts Dataset

Cargamos el archivo CSV del repositorio ekohrt/animal-fun-facts-dataset. En dond la columna text se usa como texto principal del documento. Las columnas animal_name, source, media_link y wikipedia_link se guardan como metadatos.


In [13]:
animal_dataset_url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"

animal_df = pd.read_csv(animal_dataset_url)
animal_df = animal_df.dropna(subset=["text"]).reset_index(drop=True)

metadata_columns = ["animal_name", "source", "media_link", "wikipedia_link"]

animal_documents = []
for _, row in animal_df.iterrows():
    metadata = {
        column: "" if pd.isna(row[column]) else str(row[column])
        for column in metadata_columns
    }
    animal_documents.append(Document(text=str(row["text"]), metadata=metadata))

print(f"Documentos cargados: {len(animal_documents)}")
display(animal_df.head())

Documentos cargados: 7731


,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


### Creación de VectorStore y carga de documentos


In [14]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

animal_vector_store = VectorStore(embedding_model)
animal_vector_store.add_documents(animal_documents)

print(f"Documentos agregados al VectorStore: {len(animal_vector_store.documents)}")
print(f"Dimensiones de embeddings: {animal_vector_store.embeddings.shape}")

Batches: 100%|██████████| 242/242 [00:08<00:00, 28.90it/s]

Documentos agregados al VectorStore: 7731
Dimensiones de embeddings: (7731, 384)


In [15]:
def results_to_dataframe(results: list[SearchResult]) -> pd.DataFrame:
    rows = []
    for result in results:
        rows.append({
            "score": round(result.score, 4),
            "text": result.document.text,
            "metadata": result.document.metadata
        })
    return pd.DataFrame(rows)


def show_query_results(store, query: str, top_k: int = 3, metadata_filter: dict[str, str] | None = None):
    print("Query:", query)
    if metadata_filter is not None:
        print("Filtro:", metadata_filter)
        results = store.search(query=query, top_k=top_k, metadata_filter=metadata_filter)
    else:
        results = store.search(query=query, top_k=top_k)
    display(results_to_dataframe(results))

### 5 consultas de ejemplo con VectorStore


In [16]:
animal_queries = [
    "animals that can fly at night",
    "animals with powerful claws for digging",
    "facts about animals that live in the ocean",
    "fastest land animal",
    "animals that have unusual teeth"
]

for query in animal_queries:
    show_query_results(animal_vector_store, query, top_k=3)
    print("-" * 100)

Query: animals that can fly at night


,score,text,metadata
0,0.6372,They are nocturnal..\nThey are most active at ...,"{'animal_name': 'sydney funnel-web spider', 's..."
1,0.6333,"They are strictly nocturnal, keeping out of th...","{'animal_name': 'leopard gecko', 'source': 'ht..."
2,0.6333,"They are strictly nocturnal, keeping out of th...","{'animal_name': 'leopard gecko', 'source': 'ht..."


----------------------------------------------------------------------------------------------------
Query: animals with powerful claws for digging


,score,text,metadata
0,0.7220,These dogs are known for their fast digging ab...,"{'animal_name': 'smooth fox terrier', 'source'..."
1,0.6766,Echidnas are powerful diggers.\nThese unusual ...,"{'animal_name': 'echidna', 'source': 'https://..."
2,0.6724,Anteaters walk on their balled-up fists to kee...,"{'animal_name': 'anteater', 'source': 'https:/..."


----------------------------------------------------------------------------------------------------
Query: facts about animals that live in the ocean


...

----------------------------------------------------------------------------------------------------
Query: fastest land animal


,score,text,metadata
0,0.8639,The fastest land mammal in the world!,"{'animal_name': 'cheetah', 'source': 'https://..."
1,0.8303,Fastest animal on Earth,"{'animal_name': 'peregrine falcon', 'source': ..."
2,0.7082,The second largest animal on the land!,"{'animal_name': 'white rhinoceros', 'source': ..."


----------------------------------------------------------------------------------------------------
Query: animals that have unusual teeth


,score,text,metadata
0,0.6819,Has 32 teeth including fang-like canines!,"{'animal_name': 'chimpanzee', 'source': 'https..."
1,0.6698,"They are a ‘hyper-carnivore’, as their diet is...","{'animal_name': 'polar bear', 'source': 'https..."
2,0.6673,Apex freshwater predators with fearsome teeth!,"{'animal_name': 'pike fish', 'source': 'https:..."


----------------------------------------------------------------------------------------------------


## Part II: Filtering by metadata


In [17]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def _normalize(self, vectors: np.ndarray) -> np.ndarray:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return vectors / norms

    def _matches_metadata_filter(self, document: Document, metadata_filter: dict[str, str] | None) -> bool:
        if metadata_filter is None:
            return True

        for key, expected_value in metadata_filter.items():
            actual_value = document.metadata.get(key)
            if str(actual_value) != str(expected_value):
                return False
        return True

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [document.text for document in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True
        )
        new_embeddings = self._normalize(new_embeddings.astype(np.float32))

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        candidate_indices = [
            index for index, document in enumerate(self.documents)
            if self._matches_metadata_filter(document, metadata_filter)
        ]

        if not candidate_indices:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            show_progress_bar=False
        )
        query_embedding = self._normalize(query_embedding.astype(np.float32))[0]

        candidate_embeddings = self.embeddings[candidate_indices]
        candidate_scores = candidate_embeddings @ query_embedding

        top_k = min(top_k, len(candidate_indices))
        ranked_positions = np.argsort(candidate_scores)[::-1][:top_k]

        return [
            SearchResult(
                score=float(candidate_scores[position]),
                document=self.documents[candidate_indices[position]]
            )
            for position in ranked_positions
        ]

### Dataset seleccionado para FilteredVectorStore

Para la parte con filtrado se usa el dataset AG News, un conjunto de noticias de texto natural. Cada documento combina título y descripción. Los metadatos usados son category, source_dataset, split y row_id.


In [ ]:
ag_news_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
category_map = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

try:
    news_df = pd.read_csv(
        ag_news_url,
        header=None,
        names=["class_index", "title", "description"]
    )
    news_df["category"] = news_df["class_index"].map(category_map)
    news_df["source_dataset"] = "AG News"
    news_df["split"] = "train"
except Exception as error:
    print("No se pudo descargar AG News; se usará una muestra local de respaldo.")
    print(error)
    news_df = pd.DataFrame([
        {"title": "Stocks rise after technology earnings beat expectations", "description": "Investors reacted positively to strong software and semiconductor results.", "category": "Business"},
        {"title": "Central bank signals possible interest rate cut", "description": "Markets moved higher as officials discussed inflation and growth.", "category": "Business"},
        {"title": "Championship team wins final match in overtime", "description": "The club secured the title after a dramatic goal late in the game.", "category": "Sports"},
        {"title": "Tennis player advances after straight-set victory", "description": "The athlete dominated the match with a strong serve and consistent returns.", "category": "Sports"},
        {"title": "New satellite improves climate monitoring", "description": "Scientists will use the mission to track storms, oceans and atmospheric changes.", "category": "Sci/Tech"},
        {"title": "Researchers introduce faster artificial intelligence chip", "description": "The processor is designed for efficient machine learning workloads.", "category": "Sci/Tech"},
        {"title": "World leaders meet to discuss peace agreement", "description": "Diplomats said negotiations focused on security and humanitarian support.", "category": "World"},
        {"title": "Election results reshape national government", "description": "The vote changed the balance of power after a close campaign.", "category": "World"},
    ])
    news_df["source_dataset"] = "Local fallback news dataset"
    news_df["split"] = "fallback"

news_sample = (
    news_df
    .dropna(subset=["title", "description", "category"])
    .groupby("category", group_keys=False)
    .head(300)
    .reset_index(drop=True)
)

news_sample["row_id"] = news_sample.index.astype(str)
news_sample["text"] = news_sample["title"].astype(str) + ". " + news_sample["description"].astype(str)

news_documents = []
for _, row in news_sample.iterrows():
    metadata = {
        "category": str(row["category"]),
        "source_dataset": str(row["source_dataset"]),
        "split": str(row["split"]),
        "row_id": str(row["row_id"])
    }
    news_documents.append(Document(text=str(row["text"]), metadata=metadata))

print(f"Documentos cargados para FilteredVectorStore: {len(news_documents)}")
display(news_sample[["title", "description", "category", "source_dataset", "split", "row_id"]].head())

Documentos cargados para FilteredVectorStore: 1200


,title,description,category,source_dataset,split,row_id
0,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Business,AG News,train,0
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Business,AG News,train,1
2,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Business,AG News,train,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Business,AG News,train,3
4,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...",Business,AG News,train,4


### Creación de FilteredVectorStore y carga de documentos


In [19]:
filtered_vector_store = FilteredVectorStore(embedding_model)
filtered_vector_store.add_documents(news_documents)

print(f"Documentos agregados al FilteredVectorStore: {len(filtered_vector_store.documents)}")
print(f"Dimensiones de embeddings: {filtered_vector_store.embeddings.shape}")

Batches: 100%|██████████| 38/38 [00:02<00:00, 16.85it/s]

Documentos agregados al FilteredVectorStore: 1200
Dimensiones de embeddings: (1200, 384)


### 5 consultas de ejemplo con FilteredVectorStore y filtro de metadatos


In [20]:
filtered_queries = [
    ("software companies and new technology products", {"category": "Sci/Tech"}),
    ("team wins an important game", {"category": "Sports"}),
    ("stock market earnings and investors", {"category": "Business"}),
    ("international government and diplomatic negotiations", {"category": "World"}),
    ("internet search, computers and digital services", {"category": "Sci/Tech"})
]

for query, metadata_filter in filtered_queries:
    show_query_results(filtered_vector_store, query, top_k=3, metadata_filter=metadata_filter)
    print("-" * 100)

Query: software companies and new technology products
Filtro: {'category': 'Sci/Tech'}


,score,text,metadata
0,0.4668,IBM's mainframe momentum continues. Big Blue's...,"{'category': 'Sci/Tech', 'source_dataset': 'AG..."
1,0.4630,"Lenovo revenue grows, but problems persist. Ch...","{'category': 'Sci/Tech', 'source_dataset': 'AG..."
2,0.4579,Missing June Deals Slow to Return for Software...,"{'category': 'Sci/Tech', 'source_dataset': 'AG..."


----------------------------------------------------------------------------------------------------
Query: team wins an important game
Filtro: {'category': 'Sports'}


,score,text,metadata
0,0.4187,NBA Olympians feel urgency after stunning loss...,"{'category': 'Sports', 'source_dataset': 'AG N..."
1,0.4164,US dominance an impossible dream. It has come ...,"{'category': 'Sports', 'source_dataset': 'AG N..."
2,0.3931,"Dreaming done, NBA stars awaken to harsh Olymp...","{'category': 'Sports', 'source_dataset': 'AG N..."


----------------------------------------------------------------------------------------------------
Query: stock market earnings and investors
Filtro: {'category': 'Business'}


...

----------------------------------------------------------------------------------------------------
Query: international government and diplomatic negotiations
Filtro: {'category': 'World'}


,score,text,metadata
0,0.3772,Tigers reject Sri Lanka counter proposal to re...,"{'category': 'World', 'source_dataset': 'AG Ne..."
1,0.3131,UN tries to keep up dialogue after Khartoum ir...,"{'category': 'World', 'source_dataset': 'AG Ne..."
2,0.3065,Govt likely to cut oil product duties - offici...,"{'category': 'World', 'source_dataset': 'AG Ne..."


----------------------------------------------------------------------------------------------------
Query: internet search, computers and digital services
Filtro: {'category': 'Sci/Tech'}


,score,text,metadata
0,0.3854,A Digital Doctor Treats Computer Contamination...,"{'category': 'Sci/Tech', 'source_dataset': 'AG..."
1,0.3822,European Download Services Go Mobile (Reuters)...,"{'category': 'Sci/Tech', 'source_dataset': 'AG..."
2,0.3779,Website Lets Users Scout the Red Planet from H...,"{'category': 'Sci/Tech', 'source_dataset': 'AG..."


----------------------------------------------------------------------------------------------------


## Reflexio2n

Estuvo interesante ver la búsqueda semántica desde adentro, no solo como algo que de la nada encuentra resultados. Al implementarlo paso a paso, se entendío mejor cómo un texto puede convertirse en un vector y cómo una consulta puede compararse con muchos documentos para encontrar los que tienen más sentido, aunque no usen exactamente las mismas palabras. Muy similar a un proyecto que realice hace 1 año en machine learning, en donde con base a documentos de descirpicon de trabajo y una entrada en parrafo podiamos encontrar aquellos perfiles profesionales que mejor cubrian la necesidad de la descripción de entrada.
También me pareció interesante ver el papel de los metadatos, porque hacen que la búsqueda sea más inteligente y no tan abierta. No es lo mismo buscar en todos los documentos que decirle al sistema: “primero busca solo en esta categoría y luego dame lo más parecido”. Eso hace que los resultados se sientan más útiles y mejor enfocados.
En general, me quedo con que un vector store es como una forma de darle memoria y contexto a un sistema. No solo guarda información, sino que permite encontrarla de una manera más parecida a como pensamos las personas por relación, significado y contexto, no solamente por palabras exactas.